In [1]:
library(dplyr)
library(tidyr)
library(readr)

set.seed(2026)

dates <- seq(
  from = as.Date("2016-01-01"),
  to = as.Date("2025-12-01"),
  by = "month"
)

categories <- tibble(
  series = c(
    "education",
    "healthcare",
    "human_services",
    "higher_ed",
    "corrections",
    "aging",
    "total_expenditures"
  ),
  label = c(
    "Education",
    "Healthcare",
    "Human Services",
    "Higher Education",
    "Corrections",
    "Aging",
    "Total Expenditures"
  ),
  starting_level = c(
    700,
    520,
    430,
    180,
    150,
    65,
    2200
  ),
  monthly_growth = c(
    2.8,
    3.2,
    2.2,
    0.9,
    0.7,
    0.5,
    9.0
  ),
  noise_sd = c(
    25,
    22,
    20,
    10,
    8,
    5,
    55
  )
)

monthly_data <- crossing(
  date = dates,
  categories
) %>%
  group_by(series) %>%
  mutate(
    time = row_number(),

    month = as.integer(format(date, "%m")),

    fiscal_year = if_else(
      month >= 7,
      as.integer(format(date, "%Y")) + 1L,
      as.integer(format(date, "%Y"))
    ),

    seasonal_effect = case_when(
      month %in% c(7, 8, 9) ~ 25,
      month %in% c(1, 2) ~ -18,
      month == 12 ~ 15,
      TRUE ~ 0
    ),

    expenditure =
      starting_level +
      monthly_growth * time +
      seasonal_effect +
      rnorm(n(), mean = 0, sd = noise_sd),

    expenditure = round(expenditure, 1)
  ) %>%
  ungroup() %>%
  select(
    date,
    fiscal_year,
    month,
    series,
    label,
    expenditure
  )

write_csv(
  monthly_data,
  "../data/illinois_monthly_expenditures_simulated.csv"
)

monthly_data


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘readr’ was built under R version 4.5.2”


date,fiscal_year,month,series,label,expenditure
<date>,<int>,<int>,<chr>,<chr>,<dbl>
2016-01-01,2016,1,aging,Aging,50.1
2016-01-01,2016,1,corrections,Corrections,147.9
2016-01-01,2016,1,education,Education,691.4
2016-01-01,2016,1,healthcare,Healthcare,529.9
2016-01-01,2016,1,higher_ed,Higher Education,169.3
2016-01-01,2016,1,human_services,Human Services,433.4
2016-01-01,2016,1,total_expenditures,Total Expenditures,2253.6
2016-02-01,2016,2,aging,Aging,42.6
2016-02-01,2016,2,corrections,Corrections,150.5
